In [128]:
%pwd

'c:\\Users\\ACER\\OneDrive\\Desktop\\Medical chatbot'

In [129]:
import sys
print(sys.executable)

c:\Users\ACER\anaconda3\envs\medibot\python.exe


In [130]:
import os
import warnings
warnings.filterwarnings("ignore")

load pdf

In [131]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

lode data from pdf

In [132]:
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

In [133]:
extracted_data = load_pdf_files("data")

In [134]:
extracted_data

[Document(metadata={'producer': 'calibre 3.39.1 [https://calibre-ebook.com]', 'creator': 'calibre 3.39.1 [https://calibre-ebook.com]', 'creationdate': '2019-02-22T21:40:20+00:00', 'author': 'Hill, Lewis & Perry, Leonard', 'title': "The Fruit Gardener\\'s Bible: A Complete Guide to Growing Fruits and Nuts in the Home Garden - PDFDrive.com", 'source': 'data\\FruitGarden.pdf', 'total_pages': 596, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': 'calibre 3.39.1 [https://calibre-ebook.com]', 'creator': 'calibre 3.39.1 [https://calibre-ebook.com]', 'creationdate': '2019-02-22T21:40:20+00:00', 'author': 'Hill, Lewis & Perry, Leonard', 'title': "The Fruit Gardener\\'s Bible: A Complete Guide to Growing Fruits and Nuts in the Home Garden - PDFDrive.com", 'source': 'data\\FruitGarden.pdf', 'total_pages': 596, 'page': 1, 'page_label': '2'}, page_content='The\tFruit\tGardener’s\tBible'),
 Document(metadata={'producer': 'calibre 3.39.1 [https://calibre-ebook.com]', '

In [135]:
len(extracted_data) 

704

filter opretion cause we dont need all thing in this pdf

In [136]:
from typing import List
from langchain.docstore.document import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
   
    return [
        Document(page_content=doc.page_content, metadata={"source": doc.metadata.get("source")})
        for doc in docs
    ]

In [137]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [138]:
minimal_docs

[Document(metadata={'source': 'data\\FruitGarden.pdf'}, page_content=''),
 Document(metadata={'source': 'data\\FruitGarden.pdf'}, page_content='The\tFruit\tGardener’s\tBible'),
 Document(metadata={'source': 'data\\FruitGarden.pdf'}, page_content='The\tFruit\tGardener’s\tBible\nA\tComplete\tGuide\tto\tGrowing\tFruits\tand\tNuts\tin\tthe\tHome\tGarden\nLewis\tHill\tand\tLeonard\tPerry'),
 Document(metadata={'source': 'data\\FruitGarden.pdf'}, page_content='The\tmission\tof\tStorey\tPublishing\tis\tto\tserve\tour\tcustomers\tby\tpublishing\npractical\tinformation\tthat\tencourages\tpersonal\tindependence\tin\nharmony\twith\tthe\tenvironment.\nEdited\tby\tElizabeth\tP.\tStell\tand\tCarleen\tMadigan\tArt\tdirection\tand\tbook\ndesign\tby\tDan\tO.\tWilliams\tand\tCarolyn\tEckert\tText\tproduction\tby\nLiseann\tKarandisecky\tand\tJennifer\tJepson\tSmith\tFront\tcover\nphotography:\tfrom\ttop\tleft:\t\ngooseberries\n\t©\tJoshua\tMcCullough;\nstrawberries\n\t©\t\nbravo1954/iStockphoto.com\n;\t\

spliting the document into chunks

In [139]:
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    texts_chunks = text_splitter.split_documents(minimal_docs)
    return texts_chunks


In [140]:
texts_chunks = text_split(minimal_docs)
print(f"Number of text chunks: {len(texts_chunks)}")

Number of text chunks: 2137


In [141]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embedding = download_embeddings()

sentance transformal/all miniLM-L6-V2

In [142]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [143]:
from dotenv import load_dotenv
load_dotenv(".env")

True

In [144]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [145]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key) #it shuold be authinticated with your pinecone api key or account

In [146]:
pc

In [147]:
from pinecone import ServerlessSpec 

index_name = "chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,  # Dimension of the embeddings
        metric= "cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )


index = pc.Index(index_name)


store our vector in pinecone vactor database

In [148]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunks,
    embedding=embedding,
    index_name=index_name
)

if i want to add any new index

In [149]:
dswith = Document(
    page_content="i am lalita.",
    metadata={"source": "jhapate"}
)

In [150]:
docsearch.add_documents(documents=[dswith])

['c92ed311-7fb4-4fc8-9d08-2446cfeeb9ba']

this is the source code that i paste it into pinecone database in search box

now we connect it to LLM

In [151]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3}) # k:3 means it will return the top 3 most relevant or similar answers

In [152]:
retrieved_docs = retriever.invoke("how i plant tomato?")
retrieved_docs

[Document(id='17bb345e-48e9-4825-8258-ac8a40ba0e15', metadata={'source': 'data\\Vegitable.pdf'}, page_content='Tomato ....................................................................................................................................... 1 \nEgg Plant (Brinjal) ....................................................................................................................... 4 \nChilli ............................................................................................................................................ 7'),
 Document(id='42b8ecad-77f4-4956-b77b-57ac31571c24', metadata={'source': 'data\\Vegitable.pdf'}, page_content='Tomato ....................................................................................................................................... 1 \nEgg Plant (Brinjal) ....................................................................................................................... 4 \nChilli ....................................

In [153]:

from langchain_openai import OpenAI


chatModel = OpenAI(
    model="gpt-4o",
    temperature=0.7
)


In [158]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [155]:
system_prompt = (
    "You are an expert gardener and agricultural advisor. "
    "You have access to a collection of PDFs containing detailed information about fruit and vegetable gardening. "

    "Your goal is to: "
    "1. Answer user questions about planting, care, soil, compost, watering, sunlight, pests, and harvesting. "
    "2. Provide clear, step-by-step guidance for beginners and advanced gardeners alike. "
    "3. Use only the information available in the provided PDFs, but if the answer can be logically inferred, do so. "
    "4. Be concise, practical, and encouraging in your advice."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [156]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)